In [ ]:
import warnings
warnings.filterwarnings('ignore')

from lambeq import AtomicType, Reader
from lambeq.backend.grammar import Box, Id, Word


N = AtomicType.NOUN

class CombReader(Reader):
    def sentence2diagram(self, sentence):
        words = Id().tensor(*[Word(w, N) for w in sentence.split()])
        layer = Box('LAYER', words.cod, N)
        return words >> layer

diagram = CombReader().sentence2diagram('John gave Mary a flower')
diagram.draw()

In [ ]:
Id().tensor(*[Word(w, N) for w in ['John', 'gave', 'Mary', 'a', 'flower']]).draw(figsize=(5,1))

In [ ]:
from lambeq import BobcatParser


parser = BobcatParser(verbose='text')
d = parser.sentence2diagram('The food is fresh')

In [ ]:
from lambeq import AtomicType, SimpleRewriteRule


N = AtomicType.NOUN
S = AtomicType.SENTENCE
adj = N @ N.l

NOT = Box('NOT', S, S)

negation_rewrite = SimpleRewriteRule(
    cod=N.r @ S @ S.l @ N,
    template=SimpleRewriteRule.placeholder(N.r @ S @ S.l @ N) >> Id(N.r) @ NOT @ Id(S.l @ N),
    words=['is', 'was', 'has', 'have'])

In [ ]:
from lambeq import Rewriter
from lambeq.backend import draw_equation


not_d = Rewriter([negation_rewrite])(d)
draw_equation(d, not_d, symbol='->', figsize=(14, 4))

In [ ]:
from lambeq import RewriteRule


class PastRewriteRule(RewriteRule):
    mapping = {
        'is': 'was',
        'are': 'were',
        'has': 'had'
    }
    def matches(self, box):
        return box.name in self.mapping
    
    def rewrite(self, box):
        new_name = self.mapping[box.name]
        return type(box)(name=new_name, cod=box.cod)

In [ ]:
past_d = Rewriter([PastRewriteRule()])(d)
draw_equation(d, past_d, symbol='->', figsize=(9, 2))

In [ ]:
d = parser.sentence2diagram('We will go')

In [ ]:
from lambeq import CircuitAnsatz
from lambeq.backend.quantum import CX, Id, Ry


class RealAnsatz(CircuitAnsatz):
    def __init__(self, ob_map, n_layers, n_single_qubit_params = 1, discard = False):

        super().__init__(ob_map,
                         n_layers,
                         n_single_qubit_params,
                         discard,
                         [Ry, ])

    def params_shape(self, n_qubits):
        return (self.n_layers + 1, n_qubits)
    
    def circuit(self, n_qubits, params):
        circuit = Id(n_qubits)
        n_layers = params.shape[0] - 1

        for i in range(n_layers):
            syms = params[i]

            # adds a layer of Y rotations
            circuit >>= Id().tensor(*[Ry(sym) for sym in syms])

            # adds a ladder of CNOTs
            for j in range(n_qubits - 1):
                circuit >>= Id(j) @ CX @ Id(n_qubits - j - 2)

        # adds a final layer of Y rotations
        circuit >>= Id().tensor(*[Ry(sym) for sym in params[-1]])

        return circuit

In [ ]:
real_d = RealAnsatz({N: 1, S: 1}, n_layers=2)(d)
real_d.draw(figsize=(12, 10))

In [ ]:
from lambeq import TensorAnsatz, Symbol
from lambeq.backend import tensor
from lambeq.backend.tensor import Dim


class UnitSpiderAnsatz(TensorAnsatz):

    def _ar(self, functor, box):
        # step 1: obtain label
        name = self._summarise_box(box)

        # step 2: map domain and codomain
        dom, cod = functor(box.dom), functor(box.cod)

        # step 3: construct and return ansatz
        syms = Symbol(name, 2, 2)
        tnsr = tensor.Box(box.name, Dim(2), Dim(2), syms)

        return tensor.Spider(Dim(2), len(box.dom), 1) >> tnsr >> tensor.Spider(Dim(2), 1, len(box.cod))

In [ ]:
ansatz = UnitSpiderAnsatz({N: Dim(2), S: Dim(2)})
positive_d = ansatz(d)
positive_d.draw()

In [ ]:
import numpy as np


syms = sorted(positive_d.free_symbols)
sym_dict = {k: -np.ones(k.size) for k in syms}
subbed_diagram = positive_d.lambdify(*syms)(*sym_dict.values())

subbed_diagram.eval()